In [1]:
# PHASE 2 - CCSM (Cross-Centre Style Mixing), ResNet18, ALL SEEDS 42/43/44 in ONE notebook (re-running resumes; finished arms are skipped).
# Kaggle: GPU on, Internet on.  ~3.5-4 h on a T4 (12 pretrainings x ~15 min + 5 min image loading), well inside one 12 h session.
# OUTPUT: /kaggle/working/checkpoints/ccsm_resnet18_<arm>_s<seed>_best.pt  -> upload all 12 files (3 seeds x 4 arms) as a NEW dataset
#         version of peesarisathvikreddy/adjclv2-checkpoints, then run phase2_D5_ccsm_evaluation.ipynb.
!pip install -q datasets

"""
CCSM = MixStyle (feature-statistics mixing after layer1, partners drawn from a DIFFERENT centre) inside SimCLR pretraining.
Arms (all ResNet18, ImageNet init, same crops + mild colour jitter, NO HED, NO grayscale):
  control   SimCLR, no restyling
  ms        CCSM: MixStyle, cross-centre partners (p=0.5, alpha=0.1)                         <- the method
  msrand    MixStyle, partners drawn at random (ablation: does the cross-centre choice matter?)
  fms       CCSM + Fourier amplitude swap on view 2 (window 0.05, partner from a different centre)
Pretraining pool: 60k UNLABELED patches from centres {0,1,3,4} (15k each). Centre 2 (paper's primary test centre) is never used.
Dropped after smoke tests (did not help): Fourier-only, Fourier windows 0.01 / 0.1.
"""
import os, time, random, warnings
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as tvm
warnings.filterwarnings('ignore')

class Cfg:
    HF = "wltjr1007/Camelyon17-WILDS"
    SEEDS = [42, 43, 44]           # training seeds
    POOL_SEED = 42                 # the 60k pool is identical across seeds
    PRE_CENTERS = [0, 1, 3, 4]
    N_PER_CENTER = 15000
    EPOCHS = 20; BATCH = 256; LR = 3e-4; WD = 0.05; TAU = 0.1
    FOURIER_BETA = 0.05; LAM_MAX = 1.0
    MS_P = 0.5; MS_ALPHA = 0.1
    ARMS = ['control', 'ms', 'msrand', 'fms']
    SIZE = 96
    CKPT_DIR = "/kaggle/working/checkpoints"

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def load_pool(hf):
    from tqdm.auto import tqdm
    centers = np.array(hf['center'])
    rng = np.random.RandomState(Cfg.POOL_SEED); idx = []
    for c in Cfg.PRE_CENTERS:
        idx += list(rng.choice(np.where(centers == c)[0], Cfg.N_PER_CENTER, replace=False))
    imgs = np.zeros((len(idx), Cfg.SIZE, Cfg.SIZE, 3), np.uint8)
    for k, i in enumerate(tqdm(idx, desc='loading pool')):
        im = hf[int(i)]['image'].convert('RGB')
        if im.size != (Cfg.SIZE, Cfg.SIZE): im = im.resize((Cfg.SIZE, Cfg.SIZE))
        imgs[k] = np.asarray(im)
    return torch.from_numpy(imgs).permute(0, 3, 1, 2).contiguous(), torch.tensor(centers[idx])

def base_aug(x):
    B = x.size(0)
    a = torch.empty(B, device=x.device).uniform_(0.4, 1.0); s = a.sqrt()
    tx = (torch.rand(B, device=x.device) * 2 - 1) * (1 - s); ty = (torch.rand(B, device=x.device) * 2 - 1) * (1 - s)
    fx = torch.where(torch.rand(B, device=x.device) < 0.5, -1.0, 1.0); fy = torch.where(torch.rand(B, device=x.device) < 0.5, -1.0, 1.0)
    th = torch.zeros(B, 2, 3, device=x.device); th[:, 0, 0] = s * fx; th[:, 1, 1] = s * fy; th[:, 0, 2] = tx; th[:, 1, 2] = ty
    x = F.grid_sample(x, F.affine_grid(th, x.shape, align_corners=False), padding_mode='reflection', align_corners=False)
    r = lambda lo, hi: torch.empty(B, 1, 1, 1, device=x.device).uniform_(lo, hi)
    x = x * r(0.8, 1.2)
    m = x.mean((1, 2, 3), keepdim=True); x = (x - m) * r(0.8, 1.2) + m
    g = x.mean(1, keepdim=True); x = (x - g) * r(0.8, 1.2) + g
    return x.clamp(0, 1)

def partner_index(c, cross=True):
    if not cross:
        return torch.randperm(len(c), device=c.device)
    mask = (c[:, None] != c[None, :]).float()
    mask[mask.sum(1) == 0] = 1.0
    return torch.multinomial(mask, 1).squeeze(1)

def fourier_swap(x, j, beta, lam_max):
    B, _, H, W = x.shape
    Fx = torch.fft.fft2(x.float()); amp = torch.fft.fftshift(Fx.abs(), dim=(-2, -1)); ph = Fx.angle()
    h = int(beta * H); c = H // 2; sl = slice(c - h, c + h + 1)
    lam = torch.empty(B, 1, 1, 1, device=x.device).uniform_(0, lam_max)
    new = amp.clone()
    new[..., sl, sl] = (1 - lam) * amp[..., sl, sl] + lam * amp[j][..., sl, sl]
    new = torch.fft.ifftshift(new, dim=(-2, -1))
    return torch.fft.ifft2(new * torch.exp(1j * ph)).real.clamp(0, 1)

class Net(nn.Module):
    """backbone = Sequential(conv1,bn1,relu,maxpool,layer1..4,avgpool) -> same `backbone.*` keys as the adjacency checkpoints."""
    def __init__(self, mixstyle=False):
        super().__init__()
        r = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
        self.backbone = nn.Sequential(*list(r.children())[:-1])
        self.head = nn.Sequential(nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 128))
        self.mixstyle = mixstyle; self.mix_idx = None
    def feats(self, x):
        for i, m in enumerate(self.backbone):
            x = m(x)
            if i == 4 and self.mixstyle and self.training and self.mix_idx is not None and random.random() < Cfg.MS_P:   # after layer1
                mu = x.mean((2, 3), keepdim=True); sd = (x.var((2, 3), keepdim=True) + 1e-6).sqrt()
                lam = torch.distributions.Beta(Cfg.MS_ALPHA, Cfg.MS_ALPHA).sample((x.size(0), 1, 1, 1)).to(x.device)
                j = self.mix_idx
                x = (x - mu) / sd * (lam * sd + (1 - lam) * sd[j]) + (lam * mu + (1 - lam) * mu[j])
        return torch.flatten(x, 1)
    def forward(self, x):
        return F.normalize(self.head(self.feats(x)), dim=1)

def nt_xent(z, tau):
    n = z.size(0) // 2
    sim = z @ z.t() / tau; sim.fill_diagonal_(-1e9)
    tgt = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)
    return F.cross_entropy(sim, tgt), (sim.argmax(1) == tgt).float().mean().item()

def normalize(x): return (x - MEAN.to(x.device)) / STD.to(x.device)

def pretrain(arm, imgs, cen, seed):
    set_seed(seed)
    ms = arm in ('ms', 'msrand', 'fms'); use_f = arm == 'fms'; cross = arm != 'msrand'
    net = Net(mixstyle=ms).to(dev).train()
    opt = torch.optim.AdamW(net.parameters(), lr=Cfg.LR, weight_decay=Cfg.WD)
    spe = len(imgs) // Cfg.BATCH
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, Cfg.LR, total_steps=Cfg.EPOCHS * spe, pct_start=0.1)
    scaler = torch.cuda.amp.GradScaler(enabled=dev.type == 'cuda'); chance = float(np.log(2 * Cfg.BATCH - 1))
    for ep in range(Cfg.EPOCHS):
        perm = torch.randperm(len(imgs)); L, A = 0.0, 0.0
        for b in range(spe):
            ix = perm[b * Cfg.BATCH:(b + 1) * Cfg.BATCH]
            x = imgs[ix].to(dev).float() / 255; c = cen[ix].to(dev)
            v1, v2 = base_aug(x), base_aug(x)
            if use_f:
                v2 = fourier_swap(v2, partner_index(c), Cfg.FOURIER_BETA, Cfg.LAM_MAX)
            if ms:
                net.mix_idx = partner_index(torch.cat([c, c]), cross)
            with torch.autocast(dev.type, enabled=dev.type == 'cuda'):
                z = net(normalize(torch.cat([v1, v2])))
            loss, acc = nt_xent(z.float(), Cfg.TAU)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            L += loss.item(); A += acc
        print(f"  [{arm} s{seed}] ep {ep + 1:2d}/{Cfg.EPOCHS} loss {L / spe:.3f} (chance {chance:.2f}) inst-acc {A / spe:.3f}")
    if L / spe > 0.95 * chance:
        print(f"  !! [{arm}] loss still at chance -> do NOT use this arm")
    return net.eval(), L / spe

def main(hf=None, tensors=None):
    imgs, cen = tensors if tensors is not None else load_pool(hf)
    os.makedirs(Cfg.CKPT_DIR, exist_ok=True)
    for seed in Cfg.SEEDS:
        for arm in Cfg.ARMS:
            path = f"{Cfg.CKPT_DIR}/ccsm_resnet18_{arm}_s{seed}_best.pt"
            if os.path.exists(path):
                print(f"  [{arm} s{seed}] already done -> skip"); continue
            t = time.time(); net, L = pretrain(arm, imgs, cen, seed)
            torch.save({'model_state_dict': net.state_dict(), 'arm': arm, 'seed': seed, 'epoch': Cfg.EPOCHS, 'best_val': L}, path)
            print(f"  [{arm} s{seed}] saved {path} | {(time.time() - t) / 60:.1f} min")
    print("\nDONE. Files in", Cfg.CKPT_DIR, ":", sorted(os.listdir(Cfg.CKPT_DIR)))

if __name__ == '__main__':
    from datasets import load_dataset, concatenate_datasets
    ds = load_dataset(Cfg.HF); parts = [ds[k] for k in ds.keys()]
    hf = concatenate_datasets(parts) if len(parts) > 1 else parts[0]
    main(hf)


README.md:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

data/train-00000-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  470MB            

data/train-00000-of-00014.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  469MB            

data/train-00001-of-00014.parquet: downloading bytes:           |  0.00B            

...

Generating test split:   0%|          | 0/85054 [00:00<?, ? examples/s]

loading pool:   0%|          | 0/60000 [00:00<?, ?it/s]

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 215MB/s]


  [control s42] ep  1/20 loss 2.213 (chance 6.24) inst-acc 0.609
  [control s42] ep  2/20 loss 0.502 (chance 6.24) inst-acc 0.954
  [control s42] ep  3/20 loss 0.317 (chance 6.24) inst-acc 0.979
  [control s42] ep  4/20 loss 0.259 (chance 6.24) inst-acc 0.986
  [control s42] ep  5/20 loss 0.230 (chance 6.24) inst-acc 0.988
  [control s42] ep  6/20 loss 0.211 (chance 6.24) inst-acc 0.991
  [control s42] ep  7/20 loss 0.197 (chance 6.24) inst-acc 0.992
  [control s42] ep  8/20 loss 0.189 (chance 6.24) inst-acc 0.992
  [control s42] ep  9/20 loss 0.175 (chance 6.24) inst-acc 0.993
  [control s42] ep 10/20 loss 0.169 (chance 6.24) inst-acc 0.994
  [control s42] ep 11/20 loss 0.160 (chance 6.24) inst-acc 0.995
  [control s42] ep 12/20 loss 0.154 (chance 6.24) inst-acc 0.995
  [control s42] ep 13/20 loss 0.148 (chance 6.24) inst-acc 0.995
  [control s42] ep 14/20 loss 0.142 (chance 6.24) inst-acc 0.996
  [control s42] ep 15/20 loss 0.140 (chance 6.24) inst-acc 0.997
  [control s42] ep 16/20 